# Phase 3 — Runtime Species Configuration from MICM

Verifies that the chemistry mechanism can be switched at runtime (namelist only,
no recompilation).  The **same binary** runs Chapman (photolysis, O₃/O/O1D)
or analytical (thermal, A→B→C) by changing `config_chemistry_mechanism`.

### Checks
| # | Test | Criterion |
|---|------|-----------|
| 1 | Species sets are mechanism-specific | Chapman has `o3, o, o1d`; analytical has `a, b, c` |
| 2 | Chapman O₃ physically reasonable | 0 < max(O₃) < 1e-3 kg/kg after 1 day |
| 3 | Analytical mass conservation | A+B+C = 1e-6 kg/kg ± 1e-10 at every timestep |
| 4 | Analytical kinetic profile | A decays, B rises, C accumulates monotonically |

**Prerequisites:**
```
scripts/run_jw_test.sh 1 chapman    → data/jw_480km_chapman/output.nc
scripts/run_jw_test.sh 1 analytical → data/jw_480km_analytical/output.nc
```

In [ ]:
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("..") / "data"
CHAPMAN_FILE = DATA_DIR / "jw_480km_chapman" / "output.nc"
ANALYTICAL_FILE = DATA_DIR / "jw_480km_analytical" / "output.nc"

assert CHAPMAN_FILE.exists(), f"Chapman output not found: {CHAPMAN_FILE}"
assert ANALYTICAL_FILE.exists(), f"Analytical output not found: {ANALYTICAL_FILE}"

ds_chap = nc.Dataset(CHAPMAN_FILE)
ds_anal = nc.Dataset(ANALYTICAL_FILE)

results = {}  # test_name → bool

## Test 1 — Species sets are mechanism-specific

Chapman mechanism activates `o3, o, o1d` tracers (MPAS packages `chem_chapman_in`).
Analytical mechanism activates `a, b, c` tracers (`chem_analytical_in`).
Neither output should contain species from the other mechanism.

In [ ]:
CHAPMAN_SPECIES = {"o3", "o", "o1d"}
ANALYTICAL_SPECIES = {"a", "b", "c"}

chap_vars = set(ds_chap.variables.keys())
anal_vars = set(ds_anal.variables.keys())

chap_has = CHAPMAN_SPECIES & chap_vars
chap_missing = CHAPMAN_SPECIES - chap_vars
chap_extra = ANALYTICAL_SPECIES & chap_vars

anal_has = ANALYTICAL_SPECIES & anal_vars
anal_missing = ANALYTICAL_SPECIES - anal_vars
anal_extra = CHAPMAN_SPECIES & anal_vars

print("Chapman output:")
print(f"  Chemistry species present: {sorted(chap_has)}")
print(f"  Missing:  {sorted(chap_missing) if chap_missing else '(none)'}")
print(f"  Unwanted: {sorted(chap_extra) if chap_extra else '(none)'}")
print()
print("Analytical output:")
print(f"  Chemistry species present: {sorted(anal_has)}")
print(f"  Missing:  {sorted(anal_missing) if anal_missing else '(none)'}")
print(f"  Unwanted: {sorted(anal_extra) if anal_extra else '(none)'}")

ok = (chap_has == CHAPMAN_SPECIES and len(chap_missing) == 0 and len(chap_extra) == 0
      and anal_has == ANALYTICAL_SPECIES and len(anal_missing) == 0 and len(anal_extra) == 0)
results["species_sets"] = ok
print(f"\n{'PASS' if ok else 'FAIL'}: Species sets are mechanism-specific")

## Test 2 — Chapman O₃ physically reasonable

After 1 day of Chapman photochemistry, O₃ mixing ratios should be positive
and within a physically meaningful range (0 < max(O₃) < 1e-3 kg/kg).

In [ ]:
o3 = ds_chap["o3"][-1, :, :]  # last timestep
o3_min = float(np.min(o3))
o3_max = float(np.max(o3))
print(f"O₃ at final timestep: min = {o3_min:.3e}, max = {o3_max:.3e} kg/kg")

ok = o3_min >= 0.0 and o3_max > 0.0 and o3_max < 1e-3
results["chapman_o3"] = ok
print(f"\n{'PASS' if ok else 'FAIL'}: Chapman O₃ physically reasonable")

## Test 3 — Analytical mass conservation

The analytical mechanism has A→B→C with equal molar masses (0.028 kg/mol).
Therefore A+B+C in kg/kg must be constant at every grid cell and timestep.
Initial condition: A = 1e-6, B = C = 0.  Tolerance: 1e-10 kg/kg.

In [ ]:
a = ds_anal["a"][:]  # (Time, nCells, nVertLevels)
b = ds_anal["b"][:]
c = ds_anal["c"][:]
total = a + b + c

init_total = 1.0e-6
max_deviation = float(np.max(np.abs(total - init_total)))
print(f"Max |A+B+C - 1e-6| across all timesteps/cells/levels: {max_deviation:.3e}")

ok = max_deviation < 1e-10
results["conservation"] = ok
print(f"\n{'PASS' if ok else 'FAIL'}: Analytical mass conservation")

## Test 4 — Analytical kinetic profile

A→B (k₁=1e-4 s⁻¹) and B→C (k₂=5e-5 s⁻¹).  Over 24 hours:
- A should decay monotonically
- B should rise (produced from A faster than consumed to C)
- C should accumulate monotonically

We also plot the time series for visual confirmation.

In [ ]:
# Take cell-mean at mid-level (level 13) for time series
mid = 13
a_ts = np.mean(a[:, :, mid], axis=1)  # (Time,)
b_ts = np.mean(b[:, :, mid], axis=1)
c_ts = np.mean(c[:, :, mid], axis=1)
nt = len(a_ts)
hours = np.arange(nt)  # output every hour for 24h

# Check monotonicity
a_decays = all(a_ts[i+1] <= a_ts[i] for i in range(nt-1))
c_grows  = all(c_ts[i+1] >= c_ts[i] for i in range(nt-1))
# B should increase over most of this 24h period (k1 > k2 and A >> B initially)
b_grows  = all(b_ts[i+1] >= b_ts[i] for i in range(nt-1))

print(f"A monotonically decays: {a_decays}")
print(f"B monotonically rises:  {b_grows}")
print(f"C monotonically grows:  {c_grows}")

ok = a_decays and b_grows and c_grows
results["kinetic_profile"] = ok
print(f"\n{'PASS' if ok else 'FAIL'}: Analytical kinetic profile")

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hours, a_ts * 1e6, "r-o", ms=3, label="A")
ax.plot(hours, b_ts * 1e6, "b-s", ms=3, label="B")
ax.plot(hours, c_ts * 1e6, "g-^", ms=3, label="C")
ax.plot(hours, (a_ts + b_ts + c_ts) * 1e6, "k--", lw=1, label="A+B+C")
ax.set_xlabel("Time [hours]")
ax.set_ylabel("Mixing ratio [×10⁻⁶ kg/kg]")
ax.set_title("Analytical mechanism: A → B → C")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

In [ ]:
ds_chap.close()
ds_anal.close()

print("=" * 50)
print("Phase 3 — Runtime Species Configuration")
print("=" * 50)
all_pass = True
for name, ok in results.items():
    tag = "PASS" if ok else "FAIL"
    print(f"  [{tag}] {name}")
    if not ok:
        all_pass = False
print("=" * 50)
print(f"Overall: {'ALL PASS' if all_pass else 'SOME FAILED'}")
assert all_pass, "Phase 3 verification failed"